Dieser Anhang beschreibt die Planung des allgemeinen Rechensystems: Entwicklungsreihenfolge, Einstellungen, Ergebnisvertrag, Module und Integration der bestehenden Funktionen. Die konkreten Untersuchungsdaten und Parameter stehen in Abschnitt 5.10, die Ergebnisse und ihre Interpretation in 5.12 und 5.13. Die Planungsabschnitte halten den damaligen Stand vor der Implementierung fest.



## Reihenfolge der technischen Entwicklung {#sec-technik-5-2}


<p>Aus den Anforderungen ergibt sich für mich folgende Reihenfolge:</p>

$$
\begin{array}{c}
\boxed{\text{1. Internes Datenformat festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{2. Datenquellen auf dieses Format abbilden}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{3. Import, Normalisierung und Validierung definieren}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{4. Veränderbare Einstellungen des Systems festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{5. Einheitlichen Ergebnisvertrag festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{6. Leeres Grundgerüst der Engine definieren}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{7. Vorhandene Funktionen integrieren}}
\end{array}
$$

<p>Die Reihenfolge ist bewusst so gewählt. Würde ich zuerst die Simulationsfunktion programmieren und erst danach entscheiden, welche Eingaben und Ausgaben sie besitzen soll, würden Datenformat, Strategien und Benutzeroberfläche unnötig voneinander abhängig.</p>
<p>Die konkrete Reihenfolge ist meine eigene Designentscheidung. Die Quellen liefern die Grundlagen dafür, welche Datenstrukturen und Finanzinformationen benötigt werden; sie schreiben nicht diese exakte Softwarearchitektur vor.</p>


## Veränderbare Einstellungen des Systems {#sec-technik-5-6}



### Allgemeine Einstellungen {#sec-technik-5-6-1}



<p>Das Modell soll nicht nur für einen einzigen fest eingebauten Versuch funktionieren. Deshalb werden veränderbare Einstellungen ausserhalb der Berechnungsfunktionen gespeichert.</p>

<p>Zu den allgemeinen Einstellungen gehören zunächst:</p>

$$
\begin{array}{l}
\textbf{Startdatum} \\
\textbf{Enddatum} \\
\textbf{Startkapital} \\
\textbf{Basiswährung} \\
\textbf{Datenfrequenz} \\
\textbf{Regel für fehlende Daten}
\end{array}
$$

<p>Diese Möglichkeiten des Systems sind noch nicht mit den späteren Werten meiner Untersuchung gleichzusetzen. Das Modell kann beispielsweise grundsätzlich verschiedene Zeiträume verarbeiten; für die Maturarbeit wird später trotzdem ein bestimmter Zeitraum festgelegt und begründet.</p>


### Strategieabhängige Einstellungen {#sec-technik-5-6-2}



<p>Auch die Strategien benötigen unterschiedliche Parameter.</p>

<p><strong>Buy-and-Hold</strong></p>

$$
\begin{array}{l}
\textbf{Verwendete Anlage} \\
\textbf{Startgewicht}
\end{array}
$$

<p><strong>Rebalancing-Portfolio</strong></p>

$$
\begin{array}{l}
\textbf{Verwendete Anlagen} \\
\textbf{Zielgewichte} \\
\textbf{Rebalancing-Rhythmus}
\end{array}
$$

<p>Damit muss die Engine technisch nicht auf genau 60/40 begrenzt sein. Die Maturarbeit kann später trotzdem 60 % Aktien und 40 % Anleihen als feste Versuchskonfiguration verwenden.</p>

<p><strong>Trendfolge</strong></p>

$$
\begin{array}{l}
\textbf{Verwendete Anlage} \\
\textbf{Kurzes SMA-Fenster} \\
\textbf{Langes SMA-Fenster} \\
\textbf{Long/Cash-Regel} \\
\textbf{Signalverzögerung} \\
\textbf{Behandlung der Cash-Phasen}
\end{array}
$$

<p>Die Signalverzögerung ist fachlich bereits durch die Vermeidung von Look-ahead Bias vorgegeben [@hilpischPythonFinanceMastering2019, S. 694]. Die konkreten SMA-Fenster werden dagegen später vor dem Hauptversuch begründet festgelegt, damit die historisch beste Kombination nicht einfach nachträglich ausgewählt wird [@hilpischPythonFinanceMastering2019, S. 697].</p>


### Konfigurationsdatei {#sec-technik-5-6-3}



<p>Für die erste Version sollen diese Einstellungen in einer JSON-Datei gespeichert werden. Im Gegensatz zu den historischen Finanzdaten sind die Einstellungen nicht einfach tabellarisch aufgebaut, sondern teilweise verschachtelt. So enthält eine Strategie beispielsweise eigene Parameter und beim Rebalancing wiederum mehrere Zielgewichte. JSON eignet sich für solche hierarchischen Strukturen besser als CSV. Zusätzlich kann Python JSON direkt einlesen und das Format lässt sich später auch von einer Weboberfläche einfach erzeugen.</p>

<p>Ein vereinfachtes Beispiel sieht so aus:</p>

In [1]:
import json

beispiel_config = {
    "period": {
        "start": "2020-01-01",
        "end": "2020-12-31"
    },
    "start_capital": 100000,
    "base_currency": "USD",
    "strategies": {
        "buy_hold": {
            "enabled": True,
            "asset": "EQ_DEMO"
        },
        "rebalance": {
            "enabled": True,
            "target_weights": {
                "EQ_DEMO": 0.60,
                "BD_DEMO": 0.40
            },
            "rebalance_frequency": "annual"
        },
        "trend": {
            "enabled": True,
            "asset": "EQ_DEMO",
            "short_window": 20,
            "long_window": 100,
            "signal_lag": 1
        }
    }
}

print(json.dumps(beispiel_config, indent=2))

{
  "period": {
    "start": "2020-01-01",
    "end": "2020-12-31"
  },
  "start_capital": 100000,
  "base_currency": "USD",
  "strategies": {
    "buy_hold": {
      "enabled": true,
      "asset": "EQ_DEMO"
    },
    "rebalance": {
      "enabled": true,
      "target_weights": {
        "EQ_DEMO": 0.6,
        "BD_DEMO": 0.4
      },
      "rebalance_frequency": "annual"
    },
    "trend": {
      "enabled": true,
      "asset": "EQ_DEMO",
      "short_window": 20,
      "long_window": 100,
      "signal_lag": 1
    }
  }
}


<p>Die Zahlen in diesem Beispiel dienen nur dazu, die Struktur zu zeigen. Sie sind noch keine endgültigen Parameter der Maturarbeit.</p>

<p>Der Vorteil der Konfigurationsdatei besteht darin, dass die Berechnungslogik nicht geändert werden muss, wenn später ein anderer Zeitraum oder ein anderer Parameter getestet wird. Eine grafische Oberfläche müsste später im Wesentlichen dieselben Werte erfassen und an die Engine übergeben.</p>


## Einheitliches Ergebnisformat {#sec-technik-5-7}



### Gemeinsamer Output aller Strategien {#sec-technik-5-7-1}



<p>Damit die Strategien später direkt verglichen werden können, sollen sie einen gemeinsamen Grundoutput liefern. Für jede Strategie wird deshalb eine vollständige Zeitreihe gespeichert:</p>

$$
\begin{array}{l}
\texttt{portfolio\_history.csv} \\[6pt]
\textbf{date} \\
\textbf{strategy} \\
\textbf{portfolio\_value} \\
\textbf{period\_return} \\
\textbf{drawdown}
\end{array}
$$

<p>Aus diesen Daten lassen sich anschliessend die bereits im Theorieteil definierten Kennzahlen berechnen. Dazu gehören unter anderem Gesamtrendite, annualisierte Rendite, Volatilität, Sharpe Ratio und Maximum Drawdown.</p>

<p>Die Zeitreihe bleibt zusätzlich erhalten, weil ein einzelner Endwert nicht zeigt, wie eine Strategie während des Untersuchungszeitraums verlaufen ist. Gerade Drawdowns und unterschiedliche Marktphasen sollen später Teil der Analyse werden [@kommerSouveraenInvestierenMit2025, S. 143–166].</p>


### Strategieabhängige Resultate {#sec-technik-5-7-2}



<p>Neben dem gemeinsamen Grundoutput werden zusätzliche Informationen gespeichert, wenn sie für eine Strategie relevant sind.</p>

<p>Für ein Rebalancing-Portfolio:</p>

$$
\begin{array}{l}
\texttt{weights\_history.csv} \\[6pt]
\texttt{date} \;|\; \texttt{strategy} \;|\; \texttt{asset\_id} \;|\; \texttt{weight\_before} \;|\; \texttt{target\_weight} \;|\; \texttt{weight\_after}
\end{array}
$$

$$
\begin{array}{l}
\texttt{trades.csv} \\[6pt]
\texttt{date} \;|\; \texttt{strategy} \;|\; \texttt{asset\_id} \;|\; \texttt{value\_before} \;|\; \texttt{target\_value} \;|\; \texttt{transaction\_value}
\end{array}
$$

<p>Für die Trendfolge:</p>


$$
\begin{array}{l}
\texttt{signals.csv} \\[6pt]
\texttt{date} \;|\; \texttt{strategy} \;|\; \texttt{asset\_id} \;|\; \texttt{signal} \;|\; \texttt{position} \;|\; \texttt{signal\_value} \;|\; \texttt{sma\_short} \;|\; \texttt{sma\_long}
\end{array}
$$

<p>Diese zusätzlichen Daten helfen später bei der Interpretation. Wenn eine Strategie in einer bestimmten Phase besonders gut oder schlecht abschneidet, kann überprüft werden, welche Gewichtung oder welches Signal zu diesem Zeitpunkt vorlag.</p>


### Zusammenfassung und Run-Manifest {#sec-technik-5-7-3}



<p>Für einen schnellen Vergleich wird zusätzlich eine Zusammenfassung erzeugt. Sie enthält beispielsweise:</p>

$$
\begin{array}{l}
\texttt{strategy} \\
\texttt{start\_value} \\
\texttt{end\_value} \\
\texttt{total\_return} \\
\texttt{annualized\_return} \\
\texttt{annualized\_volatility} \\
\texttt{sharpe\_ratio} \\
\texttt{max\_drawdown}
\end{array}
$$

<p>Zusätzlich soll jeder Simulationslauf ein <code>run_manifest.json</code> erhalten. Darin werden die tatsächlich verwendete Konfiguration, der Datenstand, der Zeitpunkt des Laufs und später auch die verwendete Git-Version gespeichert.</p>

<p>Damit lässt sich ein Ergebnis auf die Kombination aus Daten, Parametern und Code zurückführen. Das ist besonders wichtig, wenn später mehrere Varianten des Modells getestet werden.</p>


## Grundgerüst der Simulationsengine {#sec-technik-5-8}



### Aufteilung der Aufgaben {#sec-technik-5-8-1}



<p>Die zentrale Simulationsfunktion soll nicht alle Berechnungen selbst enthalten. Stattdessen werden die Aufgaben getrennt:</p>

$$
\begin{array}{rcl}
\texttt{data} & \longrightarrow & \text{Daten einlesen, vereinheitlichen und prüfen} \\[6pt]
\texttt{strategies} & \longrightarrow & \text{Regeln einer Anlagestrategie anwenden} \\[6pt]
\texttt{analysis} & \longrightarrow & \text{Gemeinsame Kennzahlen berechnen} \\[6pt]
\texttt{export} & \longrightarrow & \text{Resultate speichern}
\end{array}
$$

<p>Dadurch muss beispielsweise eine neue Datenquelle nicht wissen, wie die Sharpe Ratio berechnet wird, und eine neue Strategie muss nicht selbst CSV-Dateien laden.</p>

<p>Diese Trennung folgt vor allem aus dem Aufbau meines eigenen Systems. Hilpisch zeigt ebenfalls, dass wiederverwendbarer Python-Code aus interaktiven Beispielen in Funktionen und Module ausgelagert werden kann [@hilpischPythonFinanceMastering2019].</p>


### Vorgesehene Projektstruktur {#sec-technik-5-8-2}



$$
\begin{array}{l}
\texttt{src/} \\
\texttt{|-- funktionen.py} \\
\texttt{|-- data/} \\
\quad \texttt{|-- normalize.py} \\
\quad \texttt{|-- validate.py} \\
\quad \texttt{`-- adapters/} \\
\texttt{|-- engine/} \\
\quad \texttt{|-- config.py} \\
\quad \texttt{|-- context.py} \\
\quad \texttt{|-- result.py} \\
\quad \texttt{`-- simulation.py} \\
\texttt{|-- strategies/} \\
\quad \texttt{|-- buy\_hold.py} \\
\quad \texttt{|-- rebalance.py} \\
\quad \texttt{|-- trend.py} \\
\quad \texttt{`-- country\_weighting.py} \\
\texttt{|-- analysis/} \\
\quad \texttt{`-- metrics.py} \\
\texttt{`-- export/} \\
\quad \texttt{`-- results.py}
\end{array}
$$

<p>Die bereits vorhandene <code>funktionen.py</code> bleibt zunächst bestehen. Sie enthält die mathematischen Bestandteile, die im Theorieteil entwickelt wurden. Erst bei der eigentlichen Integration wird entschieden, ob einzelne Funktionen einen Wrapper benötigen oder sinnvoll in ein anderes Modul verschoben werden.</p>


### Gemeinsame Schnittstelle der Strategien {#sec-technik-5-8-3}



<p>Damit die Engine unterschiedliche Strategien gleich behandeln kann, sollen sie dieselbe äussere Schnittstelle erhalten.</p>

<p>Vereinfacht könnte eine Strategie später so aufgerufen werden:</p>

<div align="center">
  <code>result = strategy.run(context, params)</code>
</div>

<p><code>context</code> enthält die für den Lauf vorbereiteten Markt-, Zins- und Makrodaten sowie Zeitraum und Startkapital. <code>params</code> enthält nur die Einstellungen der jeweiligen Strategie.</p>

<p>Das Ergebnis wird in einer gemeinsamen Struktur zurückgegeben. Die Vermögensentwicklung ist dabei Pflicht; Gewichtungen, Transaktionen oder Signale sind nur vorhanden, wenn die jeweilige Strategie sie benötigt.</p>

<p>Damit kann die spätere Auswertung alle Strategien über denselben Grundaufbau vergleichen.</p>


### Ablauf eines Simulationslaufs {#sec-technik-5-8-4}



<p>Ein vollständiger Lauf soll später in folgender Reihenfolge erfolgen:</p>

$$
\begin{array}{c}
\boxed{\text{Konfiguration laden}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Daten laden und prüfen}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Gewünschten Zeitraum vorbereiten}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Aktivierte Strategien auswählen}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Strategien ausführen}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Gemeinsame Kennzahlen berechnen}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Resultate und Manifest speichern}}
\end{array}
$$

<p>Während einer Simulation sollen keine neuen Marktdaten automatisch aus dem Internet geladen werden. Dadurch bleibt klar, mit welchem Datenstand ein bestimmtes Resultat entstanden ist.</p>


## Vorbereitung der technischen Implementierung {#sec-technik-5-9}



### Verwendung der bestehenden Funktionen {#sec-technik-5-9-1}



<p>Die Funktionen aus dem Theorieteil sollen nicht durch ein zweites, unabhängiges Berechnungssystem ersetzt werden. Sie bilden die Grundlage der späteren Engine.</p>

$$
\begin{array}{l|l}
\textbf{Vorhandene Funktion} & \textbf{Geplante Verwendung} \\
\hline
\texttt{resample\_dataframe()} & \text{Datenfrequenz anpassen} \\
\texttt{prozentuale\_aenderung()} & \text{Periodische Renditen} \\
\texttt{kumulierte\_rendite()} \,/\, \texttt{wachstumsfaktor()} & \text{Kumulierte Entwicklung} \\
\texttt{geometrisches\_mittel()} \,/\, \texttt{annualisierte\_rendite()} & \text{Langfristige Rendite} \\
\texttt{standardabweichung()} \,/\, \texttt{annualisierte\_volatilitaet()} & \text{Volatilität} \\
\texttt{drawdown()} \,/\, \texttt{maximum\_drawdown()} & \text{Verlustphasen} \\
\texttt{sharpe\_ratio()} & \text{Risikobereinigte Rendite} \\
\texttt{korrelationsmatrix()} \,/\, \texttt{portfolio\_risiko()} & \text{Portfolioanalyse} \\
\texttt{neue\_gewichtung()} \,/\, \texttt{rebalancing()} & \text{Rebalancing} \\
\texttt{buy\_and\_hold()} & \text{Passive Benchmark} \\
\texttt{trendfolge()} & \text{SMA-Strategie}
\end{array}
$$

<p>Vor der Integration muss geprüft werden, welche Datentypen diese Funktionen erwarten und welche Resultate sie zurückgeben. Falls für die neue Struktur zusätzliche Hilfsfunktionen fehlen, können sie ergänzt werden. Die bereits im Theorieteil festgelegten mathematischen Definitionen sollen dabei nicht still verändert werden.</p>


### Übergabe an Codex {#sec-technik-5-9-2}



<p>Bis zu diesem Punkt sind Datenformat, Verarbeitung, Einstellungen, Ergebnisstruktur und der grundsätzliche Ablauf des Systems beschrieben. Die eigentliche technische Zusammenführung dieser Vorgaben mit den vorhandenen Funktionen erfolgt im nächsten Schritt mit Unterstützung von Codex.</p>

<p>Codex soll dabei nicht selbst festlegen, welche finanzwirtschaftliche Methode verwendet wird. Vorgegeben werden:</p>

- der Datenvertrag,
- die Konfigurationsstruktur,
- das Ergebnisformat,
- die geplante Aufteilung des Systems,
- die vorhandenen Funktionen aus <code>funktionen.py</code>,
- sowie die bereits geschriebenen theoretischen Definitionen.

<p>Der technische Auftrag besteht anschliessend darin, diese Teile zu einem lauffähigen System zusammenzuführen und Tests mit künstlichen Daten zu erstellen. Falls eine vorhandene Funktion angepasst werden muss, soll die Änderung sichtbar dokumentiert werden.</p>


### Stand vor der Implementierung {#sec-technik-5-9-3}



<p>Vor dem eigentlichen Bau des Systems sind damit die wichtigsten Schnittstellen festgelegt. Noch nicht endgültig bestimmt sind bewusst die konkreten Wertpapiere, der Untersuchungszeitraum, die risikofreie Serie und die strategieabhängigen Hauptparameter. Diese Entscheidungen werden nach der technischen Integration festgelegt und begründet, bevor der eigentliche Hauptversuch durchgeführt wird.</p>

<p>Damit endet die Planung des allgemeinen Systems. Der nächste Schritt ist die Umsetzung und Prüfung der Engine.</p>

## Darstellungscode der Ergebnisabbildungen {#sec-abbildungs-code}

Der folgende Code liest und prüft die eingefrorenen Ergebnisdateien und definiert die Funktionen für die sieben Abbildungen in Abschnitt 5.12. Er führt keine Backtests aus. Die Originalzelle bleibt für die Abhängigkeiten des Analyse-Notebooks erhalten; hier wird ihre Codequelle ohne erneute Ausführung eingebunden.

{{< embed Analyse.ipynb#99cad7ec echo=true >}}

### Originalaufrufe der sieben Abbildungen {#sec-abbildungs-aufrufe}

```python
#| label: fig-h1-vermoegen
#| fig-cap: "Vermögensentwicklung der vier H1-Strategien"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_run_lines("H1", NAMES_H1, "H1 · Vermögensentwicklung 2010–2025")

#| label: fig-h1-drawdown
#| fig-cap: "Drawdown-Verläufe der vier H1-Strategien"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_run_lines("H1", NAMES_H1, "H1 · Drawdown-Verläufe 2010–2025", drawdown=True)

#| label: fig-h1-rendite-risiko
#| fig-cap: "Rendite und Volatilität im Hauptlauf"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_risk_return()

#| label: fig-k1-vermoegen
#| fig-cap: "Vermögensentwicklung im G7-Kontrollvergleich"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_run_lines("K1", {"country_weighting": "BIP-Gewichtung (G7)", "rebalance": "Gleichgewichtung (je 1/7)"}, "K1 · Gleiche Länder-ETFs, unterschiedliche Gewichte")

#| label: fig-z1-vermoegen
#| fig-cap: "Vermögensentwicklung im längeren US-Lauf"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_run_lines("Z1", NAMES_Z1, "Z1 · US-Vermögensentwicklung 2003–2025")

#| label: fig-z1-drawdown
#| fig-cap: "Drawdown-Verläufe im längeren US-Lauf"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_run_lines("Z1", NAMES_Z1, "Z1 · US-Drawdown-Verläufe 2003–2025", drawdown=True)

#| label: fig-sma-vermoegen
#| fig-cap: "Vermögensentwicklung der drei VT-Trendvarianten"
# Quelle der Daten: jeweiliger eingefrorener Run, siehe Text direkt unter der Grafik.
plot_sma_comparison()

```
